# Creating a table of numbers from 1 to 100

In [15]:
# Regenerate PDF with *slightly rounded corners* for digit rectangles.
# Keep: 8pt inner grid, 4pt outer border, single-digit (1–9) uses one-digit width of two-digit layout.

from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib import colors
from reportlab.lib.units import mm

W, H = A4

# Layout
margin = 15 * mm
cols = 10
rows = 10
inner_w = W - 2*margin
inner_h = H - 2*margin
cell_w = inner_w / cols
cell_h = inner_h / rows

THICKNESS_RATIO = 1.0

# ---- Digit drawing with rounded corners ----

def hroundrect(c, x_center, y_center, L, th, r):
    # Draw a horizontal rounded rectangle centered at (x_center, y_center)
    c.roundRect(x_center - L/2, y_center - th/2, L, th, r, stroke=0, fill=1)


def vroundrect(c, x_center, y_center, L, th, r):
    # Draw a vertical rounded rectangle centered at (x_center, y_center)
    c.roundRect(x_center - th/2, y_center - L/2, th, L, r, stroke=0, fill=1)


def draw_segment(c, cx, cy, size, seg_id, thickness=0.16, color=colors.black, corner_ratio=0.35):
    """Draw a 7-seg style segment with *rounded corners*.
    corner_ratio controls the corner radius relative to segment thickness.
    """
    w = size
    h = size * 1.9
    t = thickness * THICKNESS_RATIO
    seg_len_h = w * 0.75
    seg_len_v = h * 0.40

    left = cx - w/2
    right = cx + w/2
    top = cy + h/2
    bottom = cy - h/2

    c.setFillColor(color)

    # Radius is a fraction of thickness * width; clamp so it doesn't exceed half of thickness
    th_abs = t * w
    r = min(th_abs * corner_ratio, th_abs * 0.48)

    if seg_id == 0:  # top
        hroundrect(c, cx, top - (t*w)/2 - 0.01*h, seg_len_h, th_abs, r)
    elif seg_id == 1:  # upper-left
        vroundrect(c, left + (t*w)/2 + 0.03*w, cy + h*0.25, seg_len_v, th_abs, r)
    elif seg_id == 2:  # upper-right
        vroundrect(c, right - (t*w)/2 - 0.03*w, cy + h*0.25, seg_len_v, th_abs, r)
    elif seg_id == 3:  # middle
        hroundrect(c, cx, cy, seg_len_h, th_abs, r)
    elif seg_id == 4:  # lower-left
        vroundrect(c, left + (t*w)/2 + 0.03*w, cy - h*0.25, seg_len_v, th_abs, r)
    elif seg_id == 5:  # lower-right
        vroundrect(c, right - (t*w)/2 - 0.03*w, cy - h*0.25, seg_len_v, th_abs, r)
    elif seg_id == 6:  # bottom
        hroundrect(c, cx, bottom + (t*w)/2 + 0.01*h, seg_len_h, th_abs, r)


def draw_digit(c, cx, cy, size, d, color=colors.black):
    seg_map = {
        0:[0,1,2,4,5,6], 1:[], 2:[0,2,3,4,6], 3:[0,2,3,5,6], 4:[1,2,3,5],
        5:[0,1,3,5,6], 6:[0,1,3,4,5,6], 7:[0,2,5], 8:[0,1,2,3,4,5,6], 9:[0,1,2,3,5,6]
    }
    if d == 1:
        # Single vertical bar with rounded ends
        w = size
        h = size * 1.9
        th_abs = (0.18 * w) * THICKNESS_RATIO
        L = h * 0.70
        r = min(th_abs * 0.35, th_abs * 0.48)
        c.setFillColor(color)
        # vertical rounded rectangle centered
        c.roundRect(cx - th_abs/2, cy - L/2, th_abs, L, r, stroke=0, fill=1)
        return
    for seg in seg_map[d]:
        draw_segment(c, cx, cy, size, seg_id=seg, thickness=0.16* THICKNESS_RATIO, color=color, corner_ratio=0.35)


def draw_number_with_single_as_one_digit(c, x, y, number, two_digit_total_width, color=colors.black):
    s = str(number)
    n = len(s)
    wd_two_digit = two_digit_total_width / 2.2
    spacing_two_digit = wd_two_digit * 0.2

    if n == 1:
        digit_w = wd_two_digit
        total_w = digit_w
        spacing = 0
    elif n == 2:
        digit_w = wd_two_digit
        spacing = spacing_two_digit
        total_w = 2*digit_w + spacing
    else:  # 100
        digit_w = two_digit_total_width / 3.4
        spacing = digit_w * 0.2
        total_w = 3*digit_w + 2*spacing

    start_x = x - total_w/2

    for i, ch in enumerate(s):
        d = int(ch)
        cx = start_x + i*(digit_w + (spacing if n>1 else 0)) + digit_w/2
        cy = y
        draw_digit(c, cx, cy, digit_w, d, color)


# Create PDF
filename = "numbers_1_100_strobogrammatic_A4_roundedCorners.pdf"
c = canvas.Canvas(filename, pagesize=A4)

# Grid: inner 8pt, outer 4pt
c.setStrokeColor(colors.Color(0.7, 0.3, 0.5)) #HexColor("#444444")) #("#1E90FF")) #colors.black)
# Inner lines
c.setLineWidth(8.0)
for i in range(1, cols):
    x = margin + i*cell_w
    c.line(x, margin, x, margin + inner_h)
for j in range(1, rows):
    y = margin + j*cell_h
    c.line(margin, y, margin + inner_w, y)
# Outer frame
c.setLineWidth(4.0)
c.rect(margin, margin, inner_w, inner_h, stroke=1, fill=0)

# Numbers
two_digit_total = cell_w * 0.62
for r in range(rows):
    for col in range(cols):
        idx = r*cols + col + 1
        cx = margin + col*cell_w + cell_w/2
        cy = margin + (rows-1-r)*cell_h + cell_h/2
        draw_number_with_single_as_one_digit(c, cx, cy, idx, two_digit_total, color=colors.black)

c.showPage()
c.save()
filename


'numbers_1_100_strobogrammatic_A4_roundedCorners.pdf'